# Future Predictions

This notebook uses the regression model trained in notebook 05 to predict the number of road accidents
for each Italian municipality for the years 2025, 2026 and 2027.

The goal is to identify which municipalities are expected to have the highest accident counts in the future,
providing actionable insights for road safety investment decisions.

The model uses population, surface area and year as predictors,
and is trained on communes with more than 12 accidents per year, excluding the COVID-19 year 2020.

In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import sys
sys.path.append('../config')
from config import *

In [2]:
# import clustered_istat_data from clean
df_merged_clustered_istat_data=pd.read_csv(CLEAN_PATH + 'clustered_istat_data.csv')

# display df_merged_clustered_istat_data
df_merged_clustered_istat_data.head()

,REF_AREA,TIME_PERIOD,KILLINJ,ROADACC,Comune,Superficie (Kmq),Popolazione residente,Anno,ROADACC_PER_CAPITA,ROADACC_PER_KM2,ROADACC_PER_CAPITA_scaled,ROADACC_PER_KM2_scaled,cluster,cluster_label
0,1001,2001,10,5,Agliè,13.1462,2557.0,2001.0,0.001955,0.380338,-0.015349,-0.161350,0,Low Risk
1,1001,2002,10,5,Agliè,13.1462,2538.0,2002.0,0.001970,0.380338,-0.009130,-0.161350,0,Low Risk
2,1001,2003,7,4,Agliè,13.1462,2588.0,2003.0,0.001546,0.304270,-0.189448,-0.198299,0,Low Risk
3,1001,2004,13,9,Agliè,13.1462,2679.0,2004.0,0.003359,0.684608,0.581114,-0.013555,1,High Per Capita
4,1001,2005,2,2,Agliè,13.1462,2674.0,2005.0,0.000748,0.152135,-0.528304,-0.272197,0,Low Risk


In [3]:
# filter municipalities with more than 12 accidents
df_filtered = df_merged_clustered_istat_data[df_merged_clustered_istat_data['ROADACC'] > 12]

# create a copy of the original dataframe for filtering
df_filtered_covid = df_filtered

# exclude 2020 data as it is a structural outlier due to COVID-19 lockdowns
df_filtered_covid = df_filtered_covid[df_filtered_covid['TIME_PERIOD'] != 2020]

In [4]:
# create x predictor and y target variables
x_predictors=df_filtered_covid[['Popolazione residente','Superficie (Kmq)', 'TIME_PERIOD']]
y_target=df_filtered_covid['ROADACC']

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x_predictors, y_target, test_size=0.2, random_state=42)

# create a linear regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(y_pred)

[ 192.68347307   94.52462291  370.08279598 ...   29.61925151   13.67107982
 1505.11673777]


In [5]:
# get the most recent data for each comune (2024)
df_2024 = df_merged_clustered_istat_data[df_merged_clustered_istat_data['TIME_PERIOD'] == 2024][['Comune', 'Popolazione residente', 'Superficie (Kmq)']].drop_duplicates()

# display the most recent data for each comune
df_2024.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6339 entries, 23 to 190899
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Comune                 6338 non-null   object 
 1   Popolazione residente  6339 non-null   float64
 2   Superficie (Kmq)       6339 non-null   float64
dtypes: float64(2), object(1)
memory usage: 198.1+ KB


In [6]:
# create future predictions dataframe
future_years = [2025, 2026, 2027]
df_future_list = []

for year in future_years:
    df_year = df_2024.copy()
    df_year['TIME_PERIOD'] = year
    df_future_list.append(df_year)

df_future = pd.concat(df_future_list, ignore_index=True)
df_future.head()

,Comune,Popolazione residente,Superficie (Kmq),TIME_PERIOD
0,Agliè,2585.0,13.1463,2025
1,Airasca,3695.0,15.7393,2025
2,Ala di Stura,463.0,46.3316,2025
3,Albiano d'Ivrea,1624.0,11.7397,2025
4,Almese,6297.0,17.8741,2025


In [7]:
# create predictions for future years
X_future = df_future[['Popolazione residente', 'Superficie (Kmq)', 'TIME_PERIOD']]
df_future['predicted_ROADACC'] = model.predict(X_future)

# sort the dataframe by predicted values in descending order 
df_future_sorted = df_future.sort_values('predicted_ROADACC', ascending=False)

# display the top 10 comuni with the highest predicted number of accidents
df_future_sorted.head(10)

# save predictions to a CSV file
df_future_sorted.to_csv(CLEAN_PATH + 'comune with highest predicted number of accidents.csv', index=False, decimal=',')


## Future Predictions Results

Using the final regression model (filtered communes with more than 12 accidents, excluding 2020),
we predicted the number of road accidents for each Italian municipality for 2025, 2026 and 2027.

The top municipalities at risk are consistently the largest Italian cities:
Roma, Milano, Napoli and Torino lead the ranking due to their large population and surface area.

Interestingly, predicted values slightly decrease year over year for each municipality,
reflecting the long-term downward trend in road accidents that the model learned from historical data.

These predictions can support investment decisions in road safety infrastructure,
prioritizing municipalities where the highest number of accidents is expected.